# 03. Rede Neural Artificial (MLP) em PyTorch — Problema de Regressão

Este notebook implementa, treina e analisa a Rede Neural Perceptron Multicamadas (`MLP`) em PyTorch para o problema de **Regressão** (predição contínua de `actual_delivery_time_minutes`), cumprindo integralmente as Seções 1 e 2 da Atividade Prática:
1. **Protocolo Experimental e Escalonamento Duplo ($X$ e $y$):** Mesma divisão treino/teste (80% / 20%, `random_state=42`), subconjunto de validação (20% do treino), padronização das *features* (`StandardScaler` ajustado exclusivamente no treino) e padronização do *target* ($z$-score com média e desvio do treino) para estabilizar a escala dos gradientes, convertendo todas as predições de volta para minutos antes de calcular as métricas.
2. **Justificativa da Arquitetura de Saída e Função de Perda:** Camada de saída linear com 1 neurônio (sem função de ativação) combinada à perda quadrática média (`nn.MSELoss`).
3. **Treinamento Multi-Sementes (5 Sementes) com *Early Stopping*:** Avaliação da configuração de referência (*Baseline*) e da configuração *Final Otimizada* com 5 sementes independentes (`[42, 100, 2026, 7, 99]`), reportando média $\pm$ desvio padrão e curvas de perda por época.

## 1. Preparação dos Dados, Padronização ($X$ e $y$) e *DataLoaders*

Redes neurais treinadas por descida de gradiente são altamente sensíveis à escala das variáveis: atributos com magnitude elevada (como `estimated_delivery_time_minutes` $\in [10, 100]$) geram derivadas parciais desproporcionais em relação a variáveis binárias (`weather_Storm` $\in \{0, 1\}$), deformando a superfície de perda em ravinas elípticas que lentificam ou desestabilizam a otimização.

Além disso, na regressão padronizamos também a variável alvo no treinamento:
$$y_{\text{scaled}} = \frac{y - \mu_{y,\text{treino}}}{\sigma_{y,\text{treino}}}, \qquad \hat{y}_{\text{minutos}} = \hat{y}_{\text{scaled}} \cdot \sigma_{y,\text{treino}} + \mu_{y,\text{treino}}$$
Isso mantém os erros iniciais de ordem $\mathcal{O}(1)$, evitando explosão de gradientes nas primeiras épocas.

In [ ]:
from IPython.display import display
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")
from src.models import MLP, count_trainable_parameters, estimate_macs_per_sample
from src.train import (
    set_seeds,
    prepare_regression_data,
    make_dataloaders,
    build_optimizer,
    build_loss_fn,
    treinar,
    predict_regression,
    evaluate_config_seeds,
)
from src.metrics import compute_regression_metrics
from src.inference import salvar_checkpoint_completo
from src.visualization import plot_learning_curves_pair

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

data_reg = prepare_regression_data("../data/processed/dataset_regression.csv", test_size=0.20, val_size=0.20, random_state=42)

print(f"Número de features de entrada (in_dim): {data_reg['in_dim']}")
print(f"Sub-treino (64% total): {data_reg['X_tr'].shape[0]:,} amostras")
print(f"Validação  (16% total): {data_reg['X_val'].shape[0]:,} amostras")
print(f"Teste      (20% total): {data_reg['X_test'].shape[0]:,} amostras")
print(f"Estatísticas do Target no Treino -> Média: {data_reg['y_mean']:.4f} min | Desvio Padrão: {data_reg['y_std']:.4f} min")

## 2. Justificativa Técnica: Camada de Saída Linear e Função de Perda `nn.MSELoss`

* **Camada de saída linear com 1 neurônio (`nn.Linear(d, 1)` sem ativação):** Como o alvo padronizado $y_{\text{scaled}} \in \mathbb{R}$ assume valores positivos e negativos sem limite superior/inferior fixo, aplicar funções limitadas como `Sigmoid` ($[0,1]$), `Tanh` ($[-1,1]$) ou `ReLU` ($[0, +\infty)$) truncaria artificialmente as previsões abaixo da média ($y_{\text{scaled}} < 0$), impedindo a rede de prever entregas rápidas.
* **Função de perda `nn.MSELoss` (Erro Quadrático Médio):** É estritamente convexa em relação à saída linear, infinitamente diferenciável em todo o domínio (ao contrário da perda L1/MAE, cuja derivada é descontínua em zero) e penaliza erros grandes de forma proporcional à sua magnitude ($\nabla_{\hat{y}} L = \frac{2}{N}(\hat{y} - y)$), reduzindo diretamente a métrica principal do problema (RMSE).

## 3. Treinamento Multi-Sementes (5 Sementes): *Baseline* vs. *Configuração Final*

Seguindo o protocolo de reprodutibilidade (Seção 1), treinamos tanto a configuração **Baseline** (`hidden=(32, 32)`, `ReLU`, `Adam`, $\eta = 10^{-3}$, `batch_size=256`) quanto a **Configuração Final Otimizada** (`hidden=(64, 64)`, `GELU`, `AdamW`, $\eta = 10^{-3}$, `weight_decay=1e-4`, `dropout=0.05`, `batch_size=128`) com **5 sementes aleatórias distintas** (`[42, 100, 2026, 7, 99]`).

In [ ]:
SEEDS_5 = (42, 100, 2026, 7, 99)

cfg_baseline_reg = {
    "experimento": "comparacao_reg",
    "valor_testado": "Baseline (32, 32) ReLU Adam",
    "hidden": (32, 32),
    "activation": "ReLU",
    "optimizer": "Adam",
    "lr": 1e-3,
    "batch_size": 256,
    "dropout": 0.0,
    "weight_decay": 0.0,
    "n_epocas": 100,
    "paciencia": 15,
}

cfg_final_reg = {
    "experimento": "comparacao_reg",
    "valor_testado": "Final (64, 64) GELU AdamW",
    "hidden": (64, 64),
    "activation": "GELU",
    "optimizer": "AdamW",
    "lr": 1e-3,
    "batch_size": 128,
    "dropout": 0.05,
    "weight_decay": 1e-4,
    "n_epocas": 100,
    "paciencia": 15,
}

sum_base_reg, rows_base_reg, info_base_reg = evaluate_config_seeds("regression", data_reg, cfg_baseline_reg, seeds=SEEDS_5)
sum_final_reg, rows_final_reg, info_final_reg = evaluate_config_seeds("regression", data_reg, cfg_final_reg, seeds=SEEDS_5)

df_sementes_reg = pd.DataFrame(rows_base_reg + rows_final_reg)
print("=== RESULTADOS POR SEMENTE INDIVIDUAL (5 SEMENTES) — REGRESSÃO ===")
display(df_sementes_reg[["valor_testado", "seed", "n_parametros", "score_treino", "score_val", "secundaria_val_media" if "secundaria_val_media" in df_sementes_reg else "score_secundario_val", "score_terciario_val", "gap_treino_val", "melhor_epoca", "epoca_parada", "tempo_treino_s"]])

df_resumo_reg = pd.DataFrame([sum_base_reg, sum_final_reg])
df_resumo_reg.to_csv("../reports/tables/mlp_regressao_baseline_vs_final_val.csv", index=False)
print("\n=== RESUMO AGREGADO (MÉDIA ± DESVIO PADRÃO ENTRE 5 SEMENTES) — REGRESSÃO ===")
display(df_resumo_reg)

## 4. Visualização das Curvas de Perda e Ponto de Parada do *Early Stopping*

In [ ]:
plot_learning_curves_pair(
    info_base_reg["hist"],
    info_final_reg["hist"],
    task_title="MLP Regressão (Tempo de Entrega)",
    loss_label="Perda MSE (Target Padronizado)",
    filename="mlp_reg_learning_curves.png",
)

## 5. Avaliação no Conjunto de Teste e Salvamento do Checkpoint `models/modelo_regressao.pt`

Com a configuração final selecionada exclusivamente pela validação, avaliamos o modelo no conjunto de teste isolado ($N = 9.827$) na escala original em minutos e salvamos o checkpoint completo autocontido.

In [ ]:
best_model_reg = info_final_reg["best_model"]
base_model_reg = info_base_reg["best_model"]

# Predições na escala original (minutos) para Baseline e Final
y_pred_test_base = predict_regression(base_model_reg, data_reg["X_test"], data_reg["y_mean"], data_reg["y_std"])
y_pred_train_final = predict_regression(best_model_reg, data_reg["X_tr"], data_reg["y_mean"], data_reg["y_std"])
y_pred_val_final = predict_regression(best_model_reg, data_reg["X_val"], data_reg["y_mean"], data_reg["y_std"])
y_pred_test_final = predict_regression(best_model_reg, data_reg["X_test"], data_reg["y_mean"], data_reg["y_std"])

m_test_base = compute_regression_metrics(data_reg["y_test_orig"], y_pred_test_base)
m_tr_final = compute_regression_metrics(data_reg["y_tr_orig"], y_pred_train_final)
m_val_final = compute_regression_metrics(data_reg["y_val_orig"], y_pred_val_final)
m_test_final = compute_regression_metrics(data_reg["y_test_orig"], y_pred_test_final)

df_eval_reg = pd.DataFrame([
    {"Configuracao": "MLP Baseline (32, 32)", "Particao": "Teste (20%)", **m_test_base},
    {"Configuracao": "MLP Final (64, 64)", "Particao": "Treino (64%)", **m_tr_final},
    {"Configuracao": "MLP Final (64, 64)", "Particao": "Validação (16%)", **m_val_final},
    {"Configuracao": "MLP Final (64, 64)", "Particao": "Teste (20%)", **m_test_final},
])
display(df_eval_reg)

# Salvando checkpoint completo em ../models/modelo_regressao.pt
ckpt_reg = salvar_checkpoint_completo(
    caminho_pt="../models/modelo_regressao.pt",
    modelo=best_model_reg,
    scaler=data_reg["scaler"],
    colunas_features=data_reg["feature_cols"],
    task="regression",
    y_media=data_reg["y_mean"],
    y_desvio=data_reg["y_std"],
    hiperparametros_treino=cfg_final_reg,
)
size_reg_kb = os.path.getsize("../models/modelo_regressao.pt") / 1024.0
macs_reg = estimate_macs_per_sample(best_model_reg)
print(f"\nCheckpoint salvo com sucesso em '../models/modelo_regressao.pt' ({size_reg_kb:.2f} KB) | Complexidade: {macs_reg}")